# Lesson 9a: Attention — Theory

7a/7b's encoder compresses an entire input sequence into a single
fixed-size final hidden state — every word, however long the sentence,
squeezed through the same bottleneck before a decoder ever sees it. 7a's
own Jacobian-product argument already showed why this is fragile: an
early input's influence on a much later hidden state shrinks
geometrically with distance, so the "summary" a long sequence's final
state carries is disproportionately about *recent* inputs, not the
sequence as a whole. **Attention** removes the bottleneck directly:
instead of forcing everything through one fixed vector, let the decoder
look back at *every* encoder state at every step, weighted by how
relevant each one currently is. This lesson derives that mechanism from
the alignment problem it solves, through two concrete forms (additive,
then scaled dot-product), reframes it as a *differentiable soft
dictionary lookup*, and implements it from scratch, verified against
PyTorch.

By the end of this notebook you will have:
- measured **why a fixed-size encoding is a bottleneck**, using 7a's own
  Jacobian-product argument to show an early input's influence on a
  single final hidden state decaying with sequence length,
- derived **additive (Bahdanau) attention** and **scaled dot-product
  attention**, including *why* the scaling factor is $1/\sqrt{d_k}$,
- reframed attention as a **differentiable soft dictionary lookup** over
  keys and values, and
- **implemented scaled dot-product attention from scratch in NumPy**,
  verified against a PyTorch reference to floating-point precision.

## Introduction

Sequence-to-sequence models built from 7a/7b's RNN or LSTM encode a
whole input sequence by running it through the recurrence and keeping
only the *final* hidden state as a fixed-size summary — a single vector
that a decoder must then reconstruct everything relevant from. This
lesson asks, and answers with 7a's own machinery, exactly how much of
the sequence that single vector can actually be expected to retain.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (encoder weights, synthetic inputs,
# softmax-lookup queries) is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

## The Alignment Problem

A vanilla encoder-decoder feeds a source sequence through a recurrent
encoder and hands the decoder only $h_T$, the *final* hidden state, as
its entire summary of the source — every earlier position's information
must survive $T-1$ further recurrent steps to still be present there at
all. 7a derived exactly the quantity that governs how much of it
survives: $\partial h_T/\partial h_t$, a product of $T-t$ Jacobians, each
bounded by $W_{hh}$'s spectral norm. The same quantity that made a
gradient signal vanish backward through training makes an early input's
*information* vanish forward through inference — a fixed-size final
state is disproportionately a summary of *recent* positions, not the
sequence as a whole, and the problem gets strictly worse as the sequence
gets longer.

In [ ]:
def rnn_cell_forward(x_t, h_prev, Wxh, Whh, bh):
    return np.tanh(Wxh @ x_t + Whh @ h_prev + bh)


def sensitivity_to_early_state(Whh, hidden_states):
    """||dh_T/dh_1||, the operator norm of the accumulated Jacobian product
    from position 1 to position T, for every prefix length T."""
    n = Whh.shape[0]
    J = np.eye(n)
    norms = []
    for h_t in hidden_states[1:]:
        step_jacobian = np.diag(1 - h_t ** 2) @ Whh
        J = step_jacobian @ J
        norms.append(np.linalg.norm(J, ord=2))
    return np.array(norms)


rng = np.random.default_rng(SEED)
d, n, T_max = 16, 32, 60
Wxh = rng.normal(size=(n, d)) * 0.1
Whh = rng.normal(size=(n, n)) * 0.1
bh = np.zeros(n)

h = np.zeros(n)
hidden_states = [h]
for t in range(T_max):
    x_t = rng.normal(size=d) * 0.1
    h = rnn_cell_forward(x_t, h, Wxh, Whh, bh)
    hidden_states.append(h)

sensitivity = sensitivity_to_early_state(Whh, hidden_states)

plt.figure()
plt.semilogy(np.arange(1, T_max + 1), sensitivity)
plt.xlabel("sequence length after position 1 (encoder steps)")
plt.ylabel(r"$\|\partial h_T / \partial h_1\|_2$ (log scale)")
plt.title("How much of an early position survives in the final hidden state")
plt.tight_layout()
plt.show()
print(f"sensitivity after 5 steps: {sensitivity[4]:.2e}, after 60 steps: {sensitivity[-1]:.2e}")

The final hidden state's sensitivity to an early position collapses by
orders of magnitude as the sequence gets longer — exactly the fixed-size
bottleneck's problem, measured directly rather than argued informally.
Attention's fix does not try to make a single vector remember more; it
gives every future step direct, undecayed access to every encoder
state, so no information ever has to survive $T-t$ recurrent steps to
still be usable.

## Additive Attention

Bahdanau attention (2014) gives the decoder, at every output step,
access to every encoder hidden state $h_1,\dots,h_T$ directly, weighted
by a learned relevance score computed fresh at each step from the
decoder's *current* state $s$:

$$\text{score}(s, h_i) = v^\top \tanh(W_1 s + W_2 h_i), \qquad \alpha_i = \frac{\exp(\text{score}(s, h_i))}{\sum_{j=1}^{T} \exp(\text{score}(s, h_j))}, \qquad c = \sum_{i=1}^{T} \alpha_i h_i.$$

$W_1$, $W_2$ and $v$ are small, learned parameters — the score function
is itself a one-hidden-layer neural network, trained jointly with the
rest of the model to output high scores for encoder states the decoder
currently needs. The weighted sum $c$ (the **context vector**) replaces
the single fixed $h_T$ from the previous section with something
recomputed at every decoder step from *all* of $h_1,\dots,h_T$ — no
single vector ever has to carry the whole sequence's information forward
through $T$ recurrent steps, because $c$ is built fresh, directly from
every original encoder state, every time it is needed.

## Scaled Dot-Product Attention

Additive attention's score function needs its own learned parameters
($W_1$, $W_2$, $v$) and cannot be reduced to a single matrix
multiplication across all positions at once. **Scaled dot-product
attention** (Vaswani et al., 2017) replaces the small feedforward score
with a plain dot product between a **query** $q$ and each **key** $k_i$:

$$\text{score}(q, k_i) = \frac{q \cdot k_i}{\sqrt{d_k}}, \qquad \alpha_i = \text{softmax}_i(\text{score}(q, k_i)),$$

which needs no extra learned parameters beyond however $q$ and $k_i$
were produced, and — critically for actually training these at scale —
every score for every position can be computed in one matrix
multiplication, $QK^\top$, rather than one feedforward pass per pair.

**Why divide by $\sqrt{d_k}$?** If $q$ and $k$ have independent
components with mean 0 and variance 1, their dot product $q \cdot k =
\sum_{j=1}^{d_k} q_j k_j$ is a sum of $d_k$ independent, zero-mean terms,
so its variance grows **linearly with $d_k$**: $\text{Var}(q \cdot k) =
d_k$. Dividing by $\sqrt{d_k}$ rescales that variance back to
approximately 1, regardless of dimension. Without it, raw dot products
in a high-dimensional space become large in magnitude, pushing softmax's
inputs into a saturated regime where one score dominates completely and
every gradient with respect to the others is essentially zero — the
softmax analogue of a sigmoid saturating, and just as damaging to
training.

In [ ]:
def dot_product_variance(d_k, n_trials, rng):
    q = rng.normal(size=(n_trials, d_k))
    k = rng.normal(size=(n_trials, d_k))
    dots = np.sum(q * k, axis=1)
    return dots.var()


def softmax(x, axis=-1):
    shifted = x - x.max(axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=axis, keepdims=True)


rng2 = np.random.default_rng(SEED)
d_ks = [4, 16, 64, 256]
raw_variances = [dot_product_variance(d, 20000, rng2) for d in d_ks]
scaled_variances = [v / d for v, d in zip(raw_variances, d_ks)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(d_ks, raw_variances, marker="o", label="raw $q \cdot k$")
axes[0].plot(d_ks, d_ks, linestyle="--", color="gray", label="$d_k$ (theory)")
axes[0].set_xlabel("$d_k$"); axes[0].set_ylabel("variance"); axes[0].set_title("Unscaled score variance grows with $d_k$")
axes[0].legend()

# A concrete softmax-saturation check: as d_k grows, unscaled logits push
# the softmax toward one-hot; scaling keeps its entropy roughly constant.
entropies_raw, entropies_scaled = [], []
for d in d_ks:
    q = rng2.normal(size=d)
    k = rng2.normal(size=(8, d))
    raw_scores = k @ q
    scaled_scores = raw_scores / np.sqrt(d)
    p_raw, p_scaled = softmax(raw_scores), softmax(scaled_scores)
    entropies_raw.append(-(p_raw * np.log(p_raw + 1e-12)).sum())
    entropies_scaled.append(-(p_scaled * np.log(p_scaled + 1e-12)).sum())

axes[1].plot(d_ks, entropies_raw, marker="o", label="unscaled softmax entropy")
axes[1].plot(d_ks, entropies_scaled, marker="o", label="scaled softmax entropy")
axes[1].set_xlabel("$d_k$"); axes[1].set_ylabel("entropy (nats)"); axes[1].set_title("Scaling keeps softmax well-conditioned")
axes[1].legend()
plt.tight_layout()
plt.show()

Unscaled dot-product variance tracks the $d_k$ prediction almost
exactly, and the unscaled softmax's entropy collapses toward zero as
$d_k$ grows — it is picking one key almost deterministically, the
saturated regime with near-zero gradient for every other key. The scaled
version's entropy stays roughly flat across the same range of $d_k$,
which is exactly what dividing by $\sqrt{d_k}$ is designed to guarantee.

## Attention as Soft Lookup

A dictionary lookup takes a query, finds the *one* key that matches
it, and returns that key's value — a hard, discrete operation.
Attention is the differentiable relaxation of exactly this: given a
query $q$, a set of keys $k_1,\dots,k_T$ and corresponding values
$v_1,\dots,v_T$,

$$\text{Attention}(q, K, V) = \sum_{i=1}^{T} \alpha_i v_i, \qquad \alpha_i = \text{softmax}_i\!\left(\frac{q \cdot k_i}{\sqrt{d_k}}\right),$$

returns a **weighted blend** of every value, where the weights come from
how well the query matches each key — a *soft* lookup that degrades
gracefully to something close to the hard version when one key matches
much better than the rest, and blends multiple values together when
several keys match comparably well.

In [ ]:
d_k, d_v, n_items = 32, 4, 5
rng3 = np.random.default_rng(SEED)
keys = rng3.normal(size=(n_items, d_k))
values = rng3.normal(size=(n_items, d_v))


def soft_lookup(query, keys, values):
    scores = (keys @ query) / np.sqrt(keys.shape[1])
    weights = softmax(scores)
    return weights, weights @ values


# Query 1: (almost) exactly key 2 -- should behave like a hard lookup.
query_exact = keys[2] + rng3.normal(size=d_k) * 1e-3
weights_exact, result_exact = soft_lookup(query_exact, keys, values)

# Query 2: exactly halfway between two different keys -- should blend their values.
query_between = (keys[0] + keys[3]) / 2
weights_between, result_between = soft_lookup(query_between, keys, values)

print("query close to key 2, attention weights:", np.round(weights_exact, 3))
print("  matches values[2] this closely:", np.abs(result_exact - values[2]).max())
print()
print("query halfway between keys 0 and 3, attention weights:", np.round(weights_between, 3))
print("  weight on key 0:", round(weights_between[0], 3), " weight on key 3:", round(weights_between[3], 3))

A query nearly identical to one key concentrates almost all its
weight there, recovering the corresponding value to high precision — a
soft lookup behaving like a hard one when the match is unambiguous. A
query placed exactly halfway between two keys puts the great majority of
its weight on those same two keys (and almost none on the unrelated
ones) instead of arbitrarily picking a single winner, returning a
genuine blend of both values rather than an even split of all five —
the behaviour a discrete dictionary lookup has no way to express at all,
and precisely what makes attention differentiable: every $\alpha_i$
responds smoothly to small changes in the query.

## Implementation from Scratch

Every piece above composes into the single operation actually used in
practice — scaled dot-product attention over a batch of queries at
once, exactly the equation from the "Scaled Dot-Product Attention"
section, implemented directly and verified against PyTorch.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """Q: (n_q, d_k). K: (n_kv, d_k). V: (n_kv, d_v). Returns (n_q, d_v), (n_q, n_kv)."""
    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights


rng4 = np.random.default_rng(SEED)
n_q, n_kv, d_k, d_v = 6, 9, 16, 10
Q_np = rng4.normal(size=(n_q, d_k))
K_np = rng4.normal(size=(n_kv, d_k))
V_np = rng4.normal(size=(n_kv, d_v))

out_scratch, weights_scratch = scaled_dot_product_attention(Q_np, K_np, V_np)

out_torch = torch.nn.functional.scaled_dot_product_attention(
    torch.tensor(Q_np), torch.tensor(K_np), torch.tensor(V_np)
).numpy()

max_diff = np.abs(out_scratch - out_torch).max()
print(f"output shape: {out_scratch.shape}, attention weights sum to 1: {np.allclose(weights_scratch.sum(axis=-1), 1.0)}")
print(f"max abs diff vs torch.nn.functional.scaled_dot_product_attention: {max_diff:.2e}")
assert max_diff < 1e-10

The from-scratch implementation matches PyTorch's own
`scaled_dot_product_attention` to floating-point precision — every
derivation in this notebook (the score, the scaling, the softmax
weighting, the weighted sum over values) composes into exactly the
operation the Transformer architecture (10a) builds everything else
around.

## Key Takeaways

- **A fixed-size final hidden state is a measurable bottleneck**: 7a's
  own Jacobian-product argument, reused directly, shows an early
  position's influence on it decaying by orders of magnitude as the
  sequence gets longer.
- **Additive attention lets a decoder query every encoder state
  directly**, with a small learned feedforward network computing
  relevance scores fresh at every step, instead of forcing everything
  through one fixed summary vector.
- **Scaled dot-product attention replaces that feedforward score with a
  single matrix multiplication**, and the $1/\sqrt{d_k}$ scaling factor
  is not a tuning knob but a variance correction — verified directly by
  measuring unscaled dot-product variance growing linearly with $d_k$
  and unscaled softmax entropy collapsing toward zero as a result.
- **Attention is a differentiable soft dictionary lookup**: a query
  close to one key recovers close to that key's value, and a query
  between two keys returns a genuine blend — smooth in the query, unlike
  a hard lookup.
- **A from-scratch NumPy implementation of scaled dot-product attention
  matched `torch.nn.functional.scaled_dot_product_attention`** to
  floating-point precision.